In [1]:
!pip install yfinance scipy


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import minimize


portfolio = {
    "SILVERBETA.NS": {"qty": 2121, "avg_price": 198.64},
    "GOLDETF.NS": {"qty": 2950, "avg_price": 135.57},
    "HDFCSML250.NS": {"qty": 250, "avg_price": 173.13},
    "HAL.NS": {"qty": 500, "avg_price": 4278.3},
    "MAFANG.NS": {"qty": 100, "avg_price": 176.16}
}

tickers = list(portfolio.keys())


data = yf.download(tickers, start="2022-01-01", auto_adjust=True)["Close"]


if len(tickers) == 1:
    data = data.to_frame()


def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def compute_macd(series):
    ema12 = series.ewm(span=12).mean()
    ema26 = series.ewm(span=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    return macd, signal

def compute_bollinger(series):
    sma = series.rolling(20).mean()
    std = series.rolling(20).std()
    upper = sma + 2 * std
    lower = sma - 2 * std
    return upper, lower


indicators = {}

for t in tickers:
    df = pd.DataFrame(data[t]).dropna()
    df.columns = ["price"]

    df["RSI"] = compute_rsi(df["price"])
    df["MACD"], df["MACD_signal"] = compute_macd(df["price"])
    df["BB_upper"], df["BB_lower"] = compute_bollinger(df["price"])
    df["SMA_50"] = df["price"].rolling(50).mean()
    df["SMA_200"] = df["price"].rolling(200).mean()

    indicators[t] = df


returns = data.pct_change().dropna()
mean_returns = returns.mean()
cov_matrix = returns.cov()


latest_prices = data.iloc[-1]
total_value = sum(portfolio[t]["qty"] * latest_prices[t] for t in tickers)

weights = np.array([
    (portfolio[t]["qty"] * latest_prices[t]) / total_value for t in tickers
])


def portfolio_perf(weights):
    ret = np.sum(weights * mean_returns) * 252
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))
    return ret, vol

cur_ret, cur_vol = portfolio_perf(weights)

print("---- CURRENT PORTFOLIO ----")
print("Return:", round(cur_ret*100,2), "%")
print("Risk (Volatility):", round(cur_vol*100,2), "%")



def neg_sharpe(weights):
    ret, vol = portfolio_perf(weights)
    return -ret / vol

constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0,1) for _ in tickers)
init = len(tickers) * [1./len(tickers)]

opt = minimize(neg_sharpe, init, method='SLSQP',
               bounds=bounds, constraints=constraints)

opt_weights = opt.x
opt_ret, opt_vol = portfolio_perf(opt_weights)

print("\n---- OPTIMIZED PORTFOLIO ----")
for i, t in enumerate(tickers):
    print(f"{t}: {round(opt_weights[i]*100,2)}%")

print("Optimized Return:", round(opt_ret*100,2), "%")
print("Optimized Risk:", round(opt_vol*100,2), "%")




print("\n---- STOCK SIGNALS ----")

for t in tickers:
    df = indicators[t].dropna()
    latest = df.iloc[-1]

    print(f"\n{t}")
    print("RSI:", round(latest["RSI"],2))

    if latest["RSI"] > 70:
        print("→ Overbought")
    elif latest["RSI"] < 30:
        print("→ Oversold")

    if latest["MACD"] > latest["MACD_signal"]:
        print("→ Bullish MACD")
    else:
        print("→ Bearish MACD")

    if latest["price"] > latest["SMA_50"] > latest["SMA_200"]:
        print("→ Strong Uptrend")
    elif latest["price"] < latest["SMA_50"] < latest["SMA_200"]:
        print("→ Downtrend")

[*********************100%***********************]  5 of 5 completed

---- CURRENT PORTFOLIO ----
Return: 40.82 %
Risk (Volatility): 19.22 %

---- OPTIMIZED PORTFOLIO ----
SILVERBETA.NS: 44.94%
GOLDETF.NS: 11.62%
HDFCSML250.NS: 1.71%
HAL.NS: 41.73%
MAFANG.NS: 0.0%
Optimized Return: 37.89 %
Optimized Risk: 15.68 %

---- STOCK SIGNALS ----

SILVERBETA.NS
RSI: 50.27
→ Bullish MACD

GOLDETF.NS
RSI: 56.57
→ Bullish MACD
→ Strong Uptrend

HDFCSML250.NS
RSI: 52.29
→ Bearish MACD
→ Strong Uptrend

HAL.NS
RSI: 42.71
→ Bearish MACD
→ Strong Uptrend

MAFANG.NS
RSI: 50.52
→ Bearish MACD
→ Strong Uptrend



/tmp/ipykernel_2418/2805424614.py:62: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = data.pct_change().dropna()
